In [ ]:
# ============================================================================
# 🤖 개인 예산 관리 AI 에이전트 (Budget Management Agent)
# ============================================================================
# 이 프로그램은 자연어(말)로 지출과 수입을 관리할 수 있는 AI 챗봇입니다.
# 사용자: "오늘 점심으로 12,000원 썼어"
# AI: "식비로 12,000원 등록했습니다."

# ============================================================================
# 📦 필요한 라이브러리 불러오기 (Import)
# ============================================================================
import json  # JSON 파일(데이터)을 읽고 쓰기 위한 도구
import os  # 운영체제 관련 작업 (환경 변수 읽기 등)
from datetime import datetime  # 날짜와 시간을 다루는 도구
from pathlib import Path  # 파일 경로를 다루는 도구
from dotenv import load_dotenv  # .env 파일에서 설정값 읽기
from google import genai  # Google Gemini AI 를 사용하기 위한 도구

# ============================================================================
# 🔑 환경 설정 (API 키 로드)
# ============================================================================
load_dotenv()  # .env 파일에서 API 키와 모델명을 읽어옴
api_key = os.getenv("GEMINI_API_KEY")  # .env에서 API 키를 가져옴
model = os.getenv("GEMINI_MODEL", "gemini-2.0-flash")  # .env에서 사용할 AI 모델명을 가져옴
client = genai.Client(api_key=api_key)  # Gemini AI 서버와 연결하는 클라이언트 생성

# ============================================================================
# 💾 데이터 저장/로드 설정
# ============================================================================
# 데이터를 저장할 폴더 경로 설정
DATA_DIR = Path("expense_data")  # "expense_data" 폴더 경로 생성
DATA_DIR.mkdir(exist_ok=True)  # 폴더가 없으면 만들고, 있으면 그냥 놔둠

# 데이터를 저장할 파일 경로 설정
TRANSACTIONS_FILE = DATA_DIR / "transactions.json"  # 지출/수입 내역을 저장할 파일
BUDGETS_FILE = DATA_DIR / "budgets.json"  # 예산 정보를 저장할 파일


# ============================================================================
# 📂 데이터 저장/로드 함수들
# ============================================================================

def load_transactions():
    """저장된 거래 내역(지출/수입)을 파일에서 읽어옵니다."""
    # 파일이 존재하면
    if TRANSACTIONS_FILE.exists():
        # 파일을 열어서 JSON 형식의 데이터를 읽음
        with open(TRANSACTIONS_FILE, 'r', encoding='utf-8') as f:
            return json.load(f)  # 읽은 데이터를 반환
    # 파일이 없으면 빈 리스트를 반환 (처음 시작할 때)
    return []


def load_budgets():
    """저장된 예산 정보를 파일에서 읽어옵니다."""
    # 파일이 존재하면
    if BUDGETS_FILE.exists():
        # 파일을 열어서 JSON 형식의 데이터를 읽음
        with open(BUDGETS_FILE, 'r', encoding='utf-8') as f:
            return json.load(f)  # 읽은 데이터를 반환
    # 파일이 없으면 빈 사전(딕셔너리)을 반환 (처음 시작할 때)
    return {}


def save_transactions(transactions):
    """거래 내역을 파일에 저장합니다."""
    # 파일을 쓰기 모드('w')로 열고, 데이터를 JSON 형식으로 저장
    with open(TRANSACTIONS_FILE, 'w', encoding='utf-8') as f:
        # transactions(리스트)를 JSON 형식으로 변환해서 파일에 씀
        # ensure_ascii=False: 한글 글자가 깨지지 않도록 함
        # indent=2: 파일을 보기 좋게 정렬
        json.dump(transactions, f, ensure_ascii=False, indent=2)


def save_budgets(budgets):
    """예산 정보를 파일에 저장합니다."""
    # 파일을 쓰기 모드('w')로 열고, 데이터를 JSON 형식으로 저장
    with open(BUDGETS_FILE, 'w', encoding='utf-8') as f:
        # budgets(사전)을 JSON 형식으로 변환해서 파일에 씀
        json.dump(budgets, f, ensure_ascii=False, indent=2)


# ============================================================================
# 🛠️ 도구 함수들 (AI가 호출할 수 있는 기능들)
# ============================================================================

def add_expense(amount: int, category: str, date: str = None, memo: str = None) -> dict:
    """
    지출을 등록합니다.
    
    매개변수:
    - amount: 지출 금액 (예: 12000)
    - category: 지출 카테고리 (예: "식비", "교통", "쇼핑")
    - date: 지출 날짜 (예: "2026-08-31", 없으면 오늘 날짜 사용)
    - memo: 지출 내역 설명 (예: "스타벅스에서 커피")
    """
    # 날짜가 입력되지 않았으면 오늘 날짜를 사용
    if not date:
        date = datetime.now().strftime("%Y-%m-%d")  # 현재 날짜를 "YYYY-MM-DD" 형식으로
    
    # 카테고리가 비어있으면 에러 반환
    if not category:
        return {"ok": False, "error": "카테고리가 필요합니다."}
    
    # 금액이 0 이하면 에러 반환 (음수는 안 됨)
    if amount <= 0:
        return {"ok": False, "error": "금액은 0보다 커야 합니다."}
    
    # 저장된 거래 내역을 모두 읽어옴
    transactions = load_transactions()
    
    # 삭제된 ID를 피하기 위해 최대 ID에 +1
    new_id = max([t['id'] for t in transactions], default=0) + 1
    
    # 새로운 지출 정보를 사전(딕셔너리) 형식으로 생성
    new_expense = {
        "id": new_id,
        "type": "expense",  # 이것이 지출임을 표시
        "date": date,  # 지출 날짜
        "amount": amount,  # 지출 금액
        "category": category,  # 지출 카테고리
        "memo": memo or "",  # 설명 (없으면 빈 문자열)
        "created_at": datetime.now().isoformat()  # 등록한 시간
    }
    
    # 새 지출을 거래 목록에 추가
    transactions.append(new_expense)
    
    # 업데이트된 거래 목록을 파일에 저장
    save_transactions(transactions)
    
    # 성공 메시지와 등록된 지출 정보 반환
    return {
        "ok": True,
        "message": f"{category} {amount:,}원이 {date}에 등록되었습니다.",
        "expense": new_expense
    }


def add_income(amount: int, category: str, date: str = None, memo: str = None) -> dict:
    """
    수입을 등록합니다.
    
    매개변수:
    - amount: 수입 금액 (예: 2000000)
    - category: 수입 카테고리 (예: "급여", "부업", "용돈")
    - date: 수입 날짜 (예: "2026-08-31", 없으면 오늘 날짜 사용)
    - memo: 수입 내역 설명 (예: "회사 월급")
    """
    # 날짜가 입력되지 않았으면 오늘 날짜를 사용
    if not date:
        date = datetime.now().strftime("%Y-%m-%d")
    
    # 카테고리가 비어있으면 에러 반환
    if not category:
        return {"ok": False, "error": "카테고리가 필요합니다."}
    
    # 금액이 0 이하면 에러 반환
    if amount <= 0:
        return {"ok": False, "error": "금액은 0보다 커야 합니다."}
    
    # 저장된 거래 내역을 모두 읽어옴
    transactions = load_transactions()
    
    # 삭제된 ID를 피하기 위해 최대 ID에 +1
    new_id = max([t['id'] for t in transactions], default=0) + 1
    
    # 새로운 수입 정보를 사전 형식으로 생성
    new_income = {
        "id": new_id,
        "type": "income",  # 이것이 수입임을 표시
        "date": date,  # 수입 날짜
        "amount": amount,  # 수입 금액
        "category": category,  # 수입 카테고리
        "memo": memo or "",  # 설명 (없으면 빈 문자열)
        "created_at": datetime.now().isoformat()  # 등록한 시간
    }
    
    # 새 수입을 거래 목록에 추가
    transactions.append(new_income)
    
    # 업데이트된 거래 목록을 파일에 저장
    save_transactions(transactions)
    
    # 성공 메시지와 등록된 수입 정보 반환
    return {
        "ok": True,
        "message": f"[수입] {category} {amount:,}원이 {date}에 등록되었습니다.",
        "income": new_income
    }


def search_expense(category: str = None, month: str = None, date: str = None, memo: str = None) -> dict:
    """
    조건에 맞는 지출을 검색합니다.
    
    매개변수:
    - category: 특정 카테고리만 검색 (예: "식비")
    - month: 특정 월만 검색 (예: "2026-08")
    - date: 특정 날짜만 검색 (예: "2026-08-31")
    - memo: 설명에 포함된 텍스트로 검색 (예: "커피")
    """
    # 저장된 거래 내역을 모두 읽어옴
    transactions = load_transactions()
    
    # 지출만 필터링 (수입은 제외)
    results = [t for t in transactions if t.get("type") == "expense"]
    
    # 카테고리가 지정되었으면 해당 카테고리만 필터링
    if category:
        results = [t for t in results if t['category'] == category]
    
    # 월이 지정되었으면 해당 월만 필터링
    if month:
        results = [t for t in results if t['date'].startswith(month)]
    
    # 날짜가 지정되었으면 해당 날짜만 필터링
    if date:
        results = [t for t in results if t['date'] == date]
    
    # memo가 지정되었으면 그 텍스트를 포함하는 것만 필터링
    if memo:
        results = [t for t in results if memo in t.get('memo', '')]
    
    # 검색 결과가 없으면
    if not results:
        return {
            "ok": True,
            "message": "조건에 맞는 거래가 없습니다.",
            "count": 0,
            "expenses": []
        }
    
    # 검색된 지출들의 총액 계산
    total = sum(t['amount'] for t in results)
    
    # 검색 결과 반환
    return {
        "ok": True,
        "message": f"총 {len(results)}개의 거래를 찾았습니다.",
        "count": len(results),
        "total": total,
        "expenses": results
    }


def search_income(category: str = None, month: str = None, date: str = None, memo: str = None) -> dict:
    """
    조건에 맞는 수입을 검색합니다.
    
    매개변수:
    - category: 특정 카테고리만 검색 (예: "급여")
    - month: 특정 월만 검색 (예: "2026-08")
    - date: 특정 날짜만 검색 (예: "2026-08-31")
    - memo: 설명에 포함된 텍스트로 검색 (예: "월급")
    """
    # 저장된 거래 내역을 모두 읽어옴
    transactions = load_transactions()
    
    # 수입만 필터링 (지출은 제외)
    results = [t for t in transactions if t.get("type") == "income"]
    
    # 카테고리가 지정되었으면 해당 카테고리만 필터링
    if category:
        results = [t for t in results if t['category'] == category]
    
    # 월이 지정되었으면 해당 월만 필터링
    if month:
        results = [t for t in results if t['date'].startswith(month)]
    
    # 날짜가 지정되었으면 해당 날짜만 필터링
    if date:
        results = [t for t in results if t['date'] == date]
    
    # memo가 지정되었으면 그 텍스트를 포함하는 것만 필터링
    if memo:
        results = [t for t in results if memo in t.get('memo', '')]
    
    # 검색 결과가 없으면
    if not results:
        return {
            "ok": True,
            "message": "조건에 맞는 수입이 없습니다.",
            "count": 0,
            "incomes": []
        }
    
    # 검색된 수입들의 총액 계산
    total = sum(t['amount'] for t in results)
    
    # 검색 결과 반환
    return {
        "ok": True,
        "message": f"총 {len(results)}개의 수입을 찾았습니다.",
        "count": len(results),
        "total": total,
        "incomes": results
    }


def update_expense(expense_id: int, amount: int = None, category: str = None,
                   date: str = None, memo: str = None) -> dict:
    """
    지출을 수정합니다.
    
    매개변수:
    - expense_id: 수정할 지출의 ID
    - amount: 새 금액 (없으면 기존 금액 유지)
    - category: 새 카테고리 (없으면 기존 카테고리 유지)
    - date: 새 날짜 (없으면 기존 날짜 유지)
    - memo: 새 설명 (없으면 기존 설명 유지)
    """
    # 저장된 거래 내역을 모두 읽어옴
    transactions = load_transactions()
    
    # 해당 ID를 가진 지출을 찾음
    expense = next((t for t in transactions if t['id'] == expense_id and t.get("type") == "expense"), None)
    
    # 해당하는 지출이 없으면 에러 반환
    if not expense:
        return {"ok": False, "error": f"ID {expense_id}인 거래를 찾을 수 없습니다."}
    
    # 각 항목이 입력되었으면 수정 (입력되지 않은 것은 유지)
    if amount is not None:
        expense['amount'] = amount
    if category is not None:
        expense['category'] = category
    if date is not None:
        expense['date'] = date
    if memo is not None:
        expense['memo'] = memo
    
    # 수정된 거래 목록을 파일에 저장
    save_transactions(transactions)
    
    # 성공 메시지 반환
    return {
        "ok": True,
        "message": f"거래 ID {expense_id}가 수정되었습니다.",
        "expense": expense
    }


def set_budget(month: str, category: str, amount: int) -> dict:
    """
    월별 카테고리 예산을 설정합니다.
    
    매개변수:
    - month: 예산을 설정할 월 (예: "2026-08")
    - category: 카테고리 (예: "식비")
    - amount: 예산 금액 (예: 500000)
    """
    # 금액이 0 이하면 에러 반환
    if amount <= 0:
        return {"ok": False, "error": "예산은 0보다 커야 합니다."}
    
    # 저장된 예산을 모두 읽어옴
    budgets = load_budgets()
    
    # 해당 월의 예산이 없으면 새로 생성
    if month not in budgets:
        budgets[month] = {}
    
    # 해당 월과 카테고리의 예산 설정
    budgets[month][category] = {
        "budget": amount,  # 예산 금액
        "set_at": datetime.now().isoformat()  # 설정한 시간
    }
    
    # 업데이트된 예산을 파일에 저장
    save_budgets(budgets)
    
    # 성공 메시지 반환
    return {
        "ok": True,
        "message": f"{month} {category} 예산이 {amount:,}원으로 설정되었습니다.",
        "budget": budgets[month][category]
    }


def check_budget(month: str = None, category: str = None) -> dict:
    """
    예산과 지출을 비교하여 남은 예산을 확인합니다.
    
    매개변수:
    - month: 확인할 월 (예: "2026-08", 없으면 현재월)
    - category: 특정 카테고리만 확인 (없으면 전체 카테고리)
    """
    # 월이 입력되지 않았으면 현재 월을 사용
    if not month:
        month = datetime.now().strftime("%Y-%m")
    
    # 저장된 예산을 모두 읽어옴
    budgets = load_budgets()
    
    # 해당 월의 예산이 없으면 에러 메시지 반환
    if month not in budgets:
        return {
            "ok": True,
            "message": f"{month}의 예산이 설정되지 않았습니다.",
            "budget": None
        }
    
    # 해당 월의 예산 정보 가져옴
    month_budgets = budgets[month]
    
    # 해당 월의 지출 검색
    month_expenses = search_expense(month=month)['expenses']
    
    # 결과를 담을 사전 생성
    result = {
        "ok": True,
        "month": month,
        "details": []
    }
    
    # 각 카테고리별로 예산 vs 지출 비교
    for cat, budget_info in month_budgets.items():
        # 해당 카테고리의 총 지출액 계산
        spent = sum(t['amount'] for t in month_expenses if t['category'] == cat)
        
        # 예산 금액
        budget = budget_info['budget']
        
        # 남은 예산 계산
        remaining = budget - spent
        
        # 카테고리별 상세 정보 생성
        detail = {
            "category": cat,
            "budget": budget,  # 예산
            "spent": spent,  # 사용한 금액
            "remaining": remaining,  # 남은 금액
            "percent": int((spent / budget * 100)) if budget > 0 else 0  # 사용 비율 (%)
        }
        
        # 결과에 추가
        result['details'].append(detail)
    
    # 특정 카테고리만 요청했으면 그것만 필터링
    if category:
        result['details'] = [d for d in result['details'] if d['category'] == category]
    
    return result


def delete_expense(expense_id: int) -> dict:
    """
    지출을 삭제합니다.
    
    매개변수:
    - expense_id: 삭제할 지출의 ID
    """
    # 저장된 거래 내역을 모두 읽어옴
    transactions = load_transactions()
    
    # 해당 ID를 가진 지출을 찾음
    expense = next((t for t in transactions if t['id'] == expense_id and t.get("type") == "expense"), None)
    
    # 해당하는 지출이 없으면 에러 반환
    if not expense:
        return {"ok": False, "error": f"ID {expense_id}인 거래를 찾을 수 없습니다."}
    
    # 해당 ID를 제외한 거래들만 유지
    transactions = [t for t in transactions if t['id'] != expense_id]
    
    # 업데이트된 거래 목록을 파일에 저장
    save_transactions(transactions)
    
    # 성공 메시지와 삭제된 지출 정보 반환
    return {
        "ok": True,
        "message": f"거래 ID {expense_id}가 삭제되었습니다.",
        "deleted": expense
    }


def analyze_by_month(month: str = None) -> dict:
    """
    특정 월의 지출을 카테고리별로 분석합니다.
    
    매개변수:
    - month: 분석할 월 (예: "2026-08", 없으면 현재월)
    """
    # 월이 입력되지 않았으면 현재 월을 사용
    if not month:
        month = datetime.now().strftime("%Y-%m")
    
    # 해당 월의 지출 검색
    result = search_expense(month=month)
    
    # 검색 결과가 없으면
    if not result['expenses']:
        return {
            "ok": True,
            "message": f"{month}의 지출 내역이 없습니다.",
            "month": month,
            "total": 0,
            "count": 0,
            "by_category": {}
        }
    
    # 검색된 지출 정보 가져옴
    expenses = result['expenses']
    total = result['total']
    
    # 카테고리별 지출을 계산하기 위한 사전 생성
    by_category = {}
    
    # 각 지출을 카테고리별로 분류하여 합산
    for exp in expenses:
        cat = exp['category']
        
        # 해당 카테고리가 처음 나타나면 초기화
        if cat not in by_category:
            by_category[cat] = {"count": 0, "amount": 0}
        
        # 해당 카테고리의 지출 횟수와 총액 증가
        by_category[cat]["count"] += 1
        by_category[cat]["amount"] += exp['amount']
    
    # 분석 결과 반환
    return {
        "ok": True,
        "month": month,
        "total": total,  # 전체 지출액
        "count": len(expenses),  # 지출 건수
        "by_category": by_category  # 카테고리별 분석
    }


def get_category_ratio(month: str = None) -> dict:
    """
    특정 월의 카테고리별 지출 비율(%)을 계산합니다.
    
    매개변수:
    - month: 분석할 월 (예: "2026-08", 없으면 현재월)
    """
    # 월이 입력되지 않았으면 현재 월을 사용
    if not month:
        month = datetime.now().strftime("%Y-%m")
    
    # 해당 월의 지출을 카테고리별로 분석
    analysis = analyze_by_month(month)
    
    # 지출 내역이 없으면
    if not analysis['by_category']:
        return {
            "ok": True,
            "message": f"{month}의 지출 내역이 없습니다.",
            "month": month,
            "ratios": {}
        }
    
    # 전체 지출액
    total = analysis['total']
    
    # 비율을 계산하기 위한 사전 생성
    ratios = {}
    
    # 각 카테고리별로 지출 비율 계산
    for cat, data in analysis['by_category'].items():
        # 비율(%) 계산: (카테고리 지출액 / 전체 지출액) * 100
        percent = int((data['amount'] / total * 100)) if total > 0 else 0
        
        ratios[cat] = {
            "amount": data['amount'],  # 카테고리별 지출액
            "count": data['count'],  # 카테고리별 지출 건수
            "percent": percent  # 지출 비율
        }
    
    # 지출액이 많은 순서대로 정렬
    sorted_ratios = dict(sorted(ratios.items(), key=lambda x: x[1]['amount'], reverse=True))
    
    # 비율 정보 반환
    return {
        "ok": True,
        "month": month,
        "total": total,  # 전체 지출액
        "ratios": sorted_ratios  # 정렬된 카테고리별 비율
    }


# ============================================================================
# 🔧 AI 도구 정의 (Tool Schemas)
# ============================================================================
# AI가 어떤 도구를 사용할 수 있는지 정의하고, 각 도구의 매개변수를 설명함

TOOLS = [
    {
        "type": "function",
        "name": "add_expense",
        "description": "지출을 등록합니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "amount": {"type": "integer", "description": "금액 (원)"},
                "category": {"type": "string", "description": "카테고리 (식비, 교통, 쇼핑 등)"},
                "date": {"type": "string", "description": "날짜 (YYYY-MM-DD)"},
                "memo": {"type": "string", "description": "사용 내역 (예: 스타벅스에서 커피, 점심 식사 등)"}
            },
            "required": ["amount", "category", "date"]
        }
    },
    {
        "type": "function",
        "name": "add_income",
        "description": "수입을 등록합니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "amount": {"type": "integer", "description": "금액 (원)"},
                "category": {"type": "string", "description": "카테고리 (급여, 부업, 용돈 등)"},
                "date": {"type": "string", "description": "날짜 (YYYY-MM-DD)"},
                "memo": {"type": "string", "description": "수입 내역 (예: 회사 월급, 프리랜싱 등)"}
            },
            "required": ["amount", "category", "date"]
        }
    },
    {
        "type": "function",
        "name": "search_expense",
        "description": "조건에 맞는 지출을 검색합니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "category": {"type": "string", "description": "카테고리"},
                "month": {"type": "string", "description": "월 (YYYY-MM)"},
                "date": {"type": "string", "description": "날짜 (YYYY-MM-DD)"},
                "memo": {"type": "string", "description": "사용 내역 (메모에 포함된 텍스트로 검색)"}
            }
        }
    },
    {
        "type": "function",
        "name": "search_income",
        "description": "조건에 맞는 수입을 검색합니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "category": {"type": "string", "description": "카테고리"},
                "month": {"type": "string", "description": "월 (YYYY-MM)"},
                "date": {"type": "string", "description": "날짜 (YYYY-MM-DD)"},
                "memo": {"type": "string", "description": "수입 내역 (메모에 포함된 텍스트로 검색)"}
            }
        }
    },
    {
        "type": "function",
        "name": "update_expense",
        "description": "지출을 수정합니다. 먼저 search_expense로 expense_id를 찾아야 합니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "expense_id": {"type": "integer", "description": "거래 ID"},
                "amount": {"type": "integer", "description": "새 금액"},
                "category": {"type": "string", "description": "새 카테고리"},
                "date": {"type": "string", "description": "새 날짜"},
                "memo": {"type": "string", "description": "새 사용 내역"}
            },
            "required": ["expense_id"]
        }
    },
    {
        "type": "function",
        "name": "set_budget",
        "description": "월별 카테고리 예산을 설정합니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "month": {"type": "string", "description": "월 (YYYY-MM)"},
                "category": {"type": "string", "description": "카테고리"},
                "amount": {"type": "integer", "description": "예산 (원)"}
            },
            "required": ["month", "category", "amount"]
        }
    },
    {
        "type": "function",
        "name": "check_budget",
        "description": "예산과 지출을 비교합니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "month": {"type": "string", "description": "월 (YYYY-MM), 생략하면 현재월"},
                "category": {"type": "string", "description": "특정 카테고리만 조회"}
            }
        }
    },
    {
        "type": "function",
        "name": "delete_expense",
        "description": "지출을 삭제합니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "expense_id": {"type": "integer", "description": "거래 ID"}
            },
            "required": ["expense_id"]
        }
    },
    {
        "type": "function",
        "name": "analyze_by_month",
        "description": "특정 월의 지출을 카테고리별로 분석합니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "month": {"type": "string", "description": "월 (YYYY-MM), 생략하면 현재월"}
            }
        }
    },
    {
        "type": "function",
        "name": "get_category_ratio",
        "description": "특정 월의 카테고리별 지출 비율을 계산합니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "month": {"type": "string", "description": "월 (YYYY-MM), 생략하면 현재월"}
            }
        }
    }
]

# ============================================================================
# 🎯 도구 함수 맵핑 (AI가 도구 이름으로 함수를 찾을 수 있도록)
# ============================================================================
# 도구의 이름과 실제 함수를 연결하는 사전
TOOL_FUNCTIONS = {
    "add_expense": add_expense,
    "add_income": add_income,
    "search_expense": search_expense,
    "search_income": search_income,
    "update_expense": update_expense,
    "set_budget": set_budget,
    "check_budget": check_budget,
    "delete_expense": delete_expense,
    "analyze_by_month": analyze_by_month,
    "get_category_ratio": get_category_ratio,
}


# ============================================================================
# 🔌 AI 도구 실행 함수
# ============================================================================

def execute_function_call(step) -> dict:
    """
    AI가 요청한 도구를 실제로 실행합니다.
    
    과정:
    1. AI가 요청한 도구 이름을 받음
    2. TOOL_FUNCTIONS 사전에서 해당 함수를 찾음
    3. 함수를 실행하고 결과를 반환
    4. 만약 오류가 발생하면 에러 메시지를 반환
    """
    # AI가 요청한 도구 이름으로 실제 함수를 찾음
    tool_function = TOOL_FUNCTIONS.get(step.name)

    # 도구를 찾을 수 없으면 에러 반환
    if tool_function is None:
        return {"ok": False, "error": f"허용되지 않은 도구: {step.name}"}

    # 도구를 실행하고 결과를 반환
    try:
        # step.arguments는 AI가 전달한 매개변수들 (사전 형식)
        return tool_function(**step.arguments)
    except TypeError as error:
        # 매개변수의 타입이 맞지 않으면 에러
        return {"ok": False, "error": f"잘못된 인자: {error}"}
    except Exception as error:
        # 다른 모든 에러
        return {"ok": False, "error": f"도구 실행 실패: {type(error).__name__}: {error}"}


# ============================================================================
# 🤖 AI 에이전트 실행 함수
# ============================================================================

def run_expense_agent(user_input: str, max_turns: int = 5, max_calls: int = 20) -> dict:
    """
    지출 관리 AI 에이전트를 실행합니다.
    
    작동 원리:
    1. 사용자의 입력을 받음 (예: "오늘 점심으로 12,000원 썼어")
    2. Gemini AI에 전송
    3. AI가 필요한 도구를 선택하고 호출 (예: add_expense)
    4. 도구 결과를 AI에 다시 전송
    5. AI가 최종 답변을 생성하여 반환
    
    매개변수:
    - user_input: 사용자 입력
    - max_turns: 최대 반복 횟수 (무한 루프 방지)
    - max_calls: 최대 도구 호출 횟수 (무한 호출 방지)
    """
    # 첫 번째 입력은 사용자 입력
    next_input = user_input
    
    # 이전 대화의 ID (여러 번 호출할 때 문맥 유지용)
    previous_interaction_id = None
    
    # 도구 실행 로그 저장
    logs = []
    
    # 도구 호출 횟수 카운트
    function_call_count = 0

    # 현재 날짜 (AI가 오늘 날짜를 알 수 있도록)
    today = datetime.now().strftime("%Y-%m-%d")

    # 최대 5번 반복 (도구를 여러 번 호출할 수 있음)
    for turn in range(1, max_turns + 1):
        # Gemini AI에 보낼 요청 준비
        request = {
            "model": model,  # 사용할 AI 모델
            "input": next_input,  # 입력 (처음에는 사용자 입력, 그 후로는 도구 결과)
            "tools": TOOLS,  # 사용 가능한 도구 목록
            "system_instruction": (
                "당신은 개인 재정 관리 AI 에이전트입니다. "  # AI의 역할 설정
                f"오늘 날짜는 {today}입니다. "  # 현재 날짜 알려주기
                "사용자가 '오늘', '어제', '이번 주' 같은 표현을 쓰면 이 기준 날짜를 사용해서 정확한 날짜로 변환하세요. "
                "사용자의 요청을 정확히 이해하고, 필요한 도구를 사용해서 처리하세요. "
                "도구 결과를 기반으로 사용자가 이해할 수 있도록 친절하게 설명해주세요. "
                "예산이 설정되지 않았으면, 사용자에게 예산을 설정하도록 친절하게 제안해주세요."
            ),
            "store": True,  # 대화 기록 저장 (문맥 유지용)
        }

        # 이전에 호출한 적이 있으면 대화 ID를 전달 (문맥 유지)
        if previous_interaction_id is not None:
            request["previous_interaction_id"] = previous_interaction_id

        # Gemini AI에 요청 전송하고 응답 받음
        interaction = client.interactions.create(**request)
        
        # 응답에서 도구 호출 명령들을 추출
        function_calls = [step for step in interaction.steps if step.type == "function_call"]

        # 도구를 호출하지 않았으면 (AI가 최종 답변을 생성했으면)
        if not function_calls:
            # 최종 답변과 로그를 반환
            return {
                "ok": True,
                "answer": interaction.output_text,  # AI의 최종 답변
                "turns": turn,  # 반복 횟수
                "function_calls": function_call_count,  # 도구 호출 총 횟수
                "tool_logs": logs,  # 도구 호출 기록
            }

        # 다음 반복에서 AI에 보낼 입력 (도구 결과들을 모아둘 리스트)
        next_input = []
        
        # 각 도구 호출을 하나씩 실행
        for function_call in function_calls:
            function_call_count += 1  # 도구 호출 횟수 증가

            # 도구 호출이 너무 많으면 중단 (무한 루프 방지)
            if function_call_count > max_calls:
                return {
                    "ok": False,
                    "answer": None,
                    "turns": turn,
                    "tool_logs": logs,
                    "error": f"함수 호출 제한({max_calls})을 초과했습니다.",
                }

            # 도구 실행
            result = execute_function_call(function_call)
            
            # 도구 호출 기록 저장
            logs.append({
                "turn": turn,
                "tool": function_call.name,
                "arguments": function_call.arguments,
                "result": result
            })

            # 도구 결과를 AI에 전달할 형식으로 변환
            function_result = {
                "type": "function_result",
                "name": function_call.name,
                "call_id": function_call.id,
                "result": [{"type": "text", "text": json.dumps(result, ensure_ascii=False)}],
            }
            
            # 다음 반복에서 AI에 전달할 결과에 추가
            next_input.append(function_result)

        # 대화 ID 업데이트 (다음 반복에서 사용)
        previous_interaction_id = interaction.id

    # 최대 반복 횟수를 초과했을 때
    return {
        "ok": False,
        "answer": None,
        "turns": max_turns,
        "function_calls": function_call_count,
        "tool_logs": logs,
        "error": "최대 반복 횟수를 초과했습니다.",
    }

print("✅ 준비 완료! 아래 셀을 실행하세요.")

In [ ]:
print("🤖 지출 관리 에이전트에 오신 것을 환영합니다!")
print("💬 대화를 시작하세요. (종료하려면 'exit' 또는 'quit' 입력)\n")

while True:
    user_input = input("👤 당신: ").strip()

    if not user_input:
        continue

    if user_input.lower() in ["exit", "quit", "종료"]:
        print("\n👋 안녕히 가세요!")
        break

    print("\n⏳ 처리 중...\n")
    result = run_expense_agent(user_input)

    if result['ok']:
        print(f"🤖 에이전트: {result['answer']}")
        if result['tool_logs']:
            print(f"📊 도구 사용: {len(result['tool_logs'])}회")
            for log in result['tool_logs']:
                print(f"   - Turn {log['turn']}: {log['tool']}")
    else:
        print(f"❌ 오류: {result['error']}")

    print()

🤖 지출 관리 에이전트에 오신 것을 환영합니다!
💬 대화를 시작하세요. (종료하려면 'exit' 또는 'quit' 입력)


⏳ 처리 중...

🤖 에이전트: 식비를 가장 많이 사용하신 날은 **2026년 8월 5일**입니다. 

* **날짜:** 2026년 8월 5일
* **금액:** 35,000원
* **내역:** 저녁 회식

---

💡 **참고: 식비 지출 내역 전체 목록**
* 2026-08-01: 12,000원 (회사 구내식당 점심)
* **2026-08-05: 35,000원 (저녁 회식)** 🥇
* 2026-08-12: 15,000원 (베트남 쌀국수)
* 2026-08-21: 30,000원 (해물탕)

(8월 현재 총 식비 지출은 **92,000원**으로, 설정하신 식비 예산 500,000원 중 약 18.4%를 사용하셨습니다.)
📊 도구 사용: 2회
   - Turn 1: search_expense
   - Turn 2: check_budget

